In [41]:
from dotenv import load_dotenv
from pathlib import Path
import sys
import os

# Walk up until we find the project root (folder with the .env)
current_path = Path().resolve()
for parent in [current_path] + list(current_path.parents):
    if (parent / ".env").exists():
        load_dotenv(parent / ".env")
        project_root = os.getenv("PROJECT_ROOT")
        print(project_root)
        sys.path.append(project_root)     
        break


%load_ext autoreload
%autoreload 2

/mnt/home/test/beluga-call-pipeline/
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [42]:
import pandas as pd

import torch
from models.resnet import ResnetMultilabel
from models.mobilenet import MobileNetMultilabel
from models.quant_mobilenet import load_mobilenet_v3_quant

from training.cross_validation import run_cross_val, train_model


## Running the Optimization Experiments

This section covers the model optimization experiments:
- Switching from **ResNet18** to **MobileNet V3 Small**,
- Further **Truncating** the MobileNet architecture,
- Applying **8-bit Quantization-Aware Training (QAT)** on the MobileNet model.

Each experiment is executed with **cross-validation** as described in the paper.  
Results are written to a dedicated `results/` directory and subsequently examined in the `results_analysis` folder.



In [43]:
labels_df = pd.read_csv("../data/Verified_Dataset/labels/labels_merged.csv")
labels_df["ClipFilenamePt"] = labels_df["clip_filename"].str.replace(".wav", ".pt", regex=False)


label_columns = ["ECHO", "HFPC", "BBPC", "Whistle"]

data_dir = "../data"
processed_spects_dir = data_dir + "/Verified_Dataset/spectrograms/"

results_dir = "./results/new_dataset"

In [44]:
from training.cross_validation import create_test_fold_indices
labels_df = create_test_fold_indices(labels_df, 5)

TypeError: create_test_fold_indices() takes 1 positional argument but 2 were given

In [ ]:
labels_df["Site_Day"] = labels_df["Site"] + "_" + pd.to_datetime(labels_df["clip_start_time"], format="mixed").dt.strftime("%Y-%m-%d")

In [ ]:
labels_df[labels_df["Boat"]==1]["Site_Day"].unique()

<StringArray>
['BSM_2017-07-24', 'BSM_2017-07-25', 'BSM_2017-07-29', 'BSM_2017-08-01',
 'BSM_2017-08-08', 'BSM_2017-08-12', 'CAC_2021-07-14', 'CAC_2021-07-18',
 'CAC_2021-08-04', 'KAM_2020-07-26', 'KAM_2020-07-29', 'KAM_2020-07-30',
 'KAM_2020-07-31', 'KAM_2020-08-01', 'KAM_2021-07-20', 'KAM_2021-07-23',
 'KAM_2021-07-27', 'KAM_2021-07-28', 'KAM_2021-08-01', 'KAM_2021-08-05',
 'KAM_2021-08-09', 'RDL_2020-07-22', 'RDL_2020-07-30', 'RDL_2020-08-28']
Length: 24, dtype: str

In [ ]:
pd.set_option('display.max_columns', None)
labels_df.head(2)

,clip_filename,ECHO,BBPC,HFPC,Whistle,Call,Boat,labeling_effort,DETAIL,Begin Time (s),End Time (s),GROUNDTRUTH,Notes,original_filename,labeled_snippet_filename,snippet_start_time,snippet_start_s,snippet_end_s,Site,labeled_snippet_dir,HydrophoneModel,HydrophoneSensitivity,clip_start_time,clip_end_time,DETAILS,boat_labeling_file,boat_labeling_file_id,annotator,SnippetFilename,start_s,end_s,ClipFilenamePt,test_fold_idx,Site_Day
0,BSM_20170724_09471700.wav,0,0,0,0,0,1.0,evaluation_v2,NaN,0.0,1.0,a,NaN,201359382.170724093002.wav,201359382.170724094717.snippet.wav,2017-07-24 09:47:17,1035.0,1155.0,BSM,../../data/evaluation_snippets/v2/BSM_2017\Boat,201359382,-172.7,2017-07-24 09:47:17,2017-07-24 09:47:18,NaN,201359382.170724093002.Table.1.selections.txt,1.0,VA,NaN,NaN,NaN,BSM_20170724_09471700.pt,0,BSM_2017-07-24
1,BSM_20170724_09471800.wav,0,0,0,0,0,1.0,evaluation_v2,NaN,1.0,2.0,a,NaN,201359382.170724093002.wav,201359382.170724094717.snippet.wav,2017-07-24 09:47:17,1035.0,1155.0,BSM,../../data/evaluation_snippets/v2/BSM_2017\Boat,201359382,-172.7,2017-07-24 09:47:18,2017-07-24 09:47:19,NaN,201359382.170724093002.Table.1.selections.txt,1.0,VA,NaN,NaN,NaN,BSM_20170724_09471800.pt,1,BSM_2017-07-24


In [ ]:
labels_df[labels_df["labeling_effort"] == "evaluation_v1"]["labeled_snippet_filename"].value_counts(dropna=False)

labeled_snippet_filename
201359382.170724133858.snippet.wav    600
201359382.170724140912.snippet.wav    600
201359382.210714080018.snippet.wav    600
201359382.210714083053.snippet.wav    600
5725.200726190001.snippet.wav         600
5725.200726191905.snippet.wav         354
5725.200726195504.snippet.wav         247
Name: count, dtype: int64

In [ ]:
labels_df.groupby("labeled_snippet_filename")["Boat"].value_counts(dropna=False)

labeled_snippet_filename            Boat
201359382.170724094717.snippet.wav  1.0     120
201359382.170724103928.snippet.wav  1.0     120
201359382.170724133858.snippet.wav  1.0     600
201359382.170724140912.snippet.wav  0.0     600
201359382.170725060901.snippet.wav  1.0     120
201359382.210714080018.snippet.wav  0.0     600
201359382.210714083053.snippet.wav  1.0     600
201359382.210714135951.snippet.wav  0.0     116
201359382.210714142330.snippet.wav  0.0     120
201359382.210718085424.snippet.wav  1.0     120
201359382.210718085704.snippet.wav  1.0     120
201359382.210718182950.snippet.wav  0.0     116
201359382.210718192740.snippet.wav  1.0     120
201359382.210725161105.snippet.wav  0.0     120
201359382.210803184644.snippet.wav  0.0     120
201359382.210803191210.snippet.wav  0.0     120
201359382.210804123124.snippet.wav  0.0     120
201359382.210804124936.snippet.wav  0.0     120
201359382.210804132905.snippet.wav  1.0     120
5725.200726190001.snippet.wav       0.0     600

In [45]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold
from cross_validation import create_test_fold_indices

# labels_df["Boat"] = labels_df["Boat"].astype("boolean")
labels_df = create_test_fold_indices(
    labels_df, 
    n_splits=5, 
    stratify_cols=["Site", "Boat"],  # NEW: stratify on both
    group_col="labeled_snippet_filename"
)

# Check distribution
print(labels_df.groupby(['test_fold_idx', 'Boat']).size())

RuntimeError: 6805 rows were not assigned a fold (still -1).

In [27]:
labels_df["Boat"].astype("string").replace('nan', '__NA__')

0        True
1        True
2        True
3        True
4        True
         ... 
15513    <NA>
15514    <NA>
15515    <NA>
15516    <NA>
15517    <NA>
Name: Boat, Length: 15518, dtype: string

In [13]:
labels_df.groupby("test_fold_idx")["Boat"].value_counts(dropna=False)

test_fold_idx  Boat
0              NaN     1568
               0.0     1385
               1.0      151
1              NaN     1492
               0.0      868
               1.0      743
2              1.0     1226
               NaN     1139
               0.0      738
3              NaN     1135
               0.0     1079
               1.0      890
4              NaN     1471
               1.0     1345
               0.0      288
Name: count, dtype: int64

In [13]:
labels_df[labels_df["Boat"] == 1].groupby("test_fold_idx")["BBPC"].value_counts(dropna=False)

test_fold_idx  BBPC
0              0        149
               1          2
1              0        714
               1         29
2              0       1198
               1         28
3              0        855
               1         35
4              0       1272
               1         73
Name: count, dtype: int64

In [15]:
labels_df[labels_df["Boat"] == 1].groupby("labeled_snippet_filename")["BBPC"].value_counts()

labeled_snippet_filename            BBPC
201359382.170724094717.snippet.wav  0       120
201359382.170724103928.snippet.wav  0       120
201359382.170724133858.snippet.wav  0       596
                                    1         4
201359382.170725060901.snippet.wav  0       120
201359382.210714083053.snippet.wav  0       564
                                    1        36
201359382.210718085424.snippet.wav  0       120
201359382.210718085704.snippet.wav  0       120
201359382.210718192740.snippet.wav  0       120
201359382.210804132905.snippet.wav  0       117
                                    1         3
5725.200726191905.snippet.wav       0       335
                                    1        19
5725.200726195504.snippet.wav       0       229
                                    1        18
5725.200729102125.snippet.wav       0       120
5725.200729113050.snippet.wav       0       120
5725.200730122111.snippet.wav       0       119
                                    1         1